# H&M Personalized Fashion Recommendations

Multi-strategy retrieval → interaction features → LightGBM binary classifier.
The implementation lives in `src/hm_reco/`; this notebook walks through it step by step.
The same runs are available from the command line: `python train.py cv` / `python train.py submit --rounds N`.

In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from hm_reco import data, retrieval, pipeline, evaluation, submission

ds = data.load_dataset("data")   # first run parses the CSVs and caches parquet files
print(f"{len(ds.transactions):,} transactions, {len(ds.customers):,} customers, {ds.n_weeks} weeks")

## 1. Retrieval

For a target week, every strategy only sees the weeks before it. Here: how many of the last week's
purchases each source recovers.

In [ ]:
VAL_WEEK = ds.n_weeks - 1
week = pipeline.prepare_week(ds, VAL_WEEK)
labels = pipeline._labels(ds, VAL_WEEK)
buyers = ds.customers[ds.customers["customer_id"].isin(labels["customer_id"].unique())]

cand = retrieval.generate(week.ctx, buyers, ds.articles)
cand = cand.merge(labels, on=["customer_id", "article_id"], how="left")
print(f"{len(cand) / len(buyers):.0f} candidates per customer, "
      f"{int(cand['label'].sum()):,} of {len(labels):,} purchases retrieved")

sources = {"repurchase": "rep_days_ago", "itemcf": "cf_score", "siblings": "sib_sales",
           "popular": "pop_rank", "age popular": "agepop_rank"}
pd.DataFrame({name: {"candidates": int(cand[col].notna().sum()),
                     "hits": int(cand.loc[cand[col].notna(), "label"].sum())}
              for name, col in sources.items()}).T

## 2. Validation

Train on the 6 weeks before the last week (negatives downsampled), validate on the last week
(2020-09-16 .. 09-22), which mirrors the test week.

In [ ]:
cfg = pipeline.Config(n_train_weeks=6, neg_per_week=600_000)
model, score = pipeline.run_cv(ds, cfg)

In [ ]:
importance = pd.Series(model.feature_importance("gain"), index=model.feature_name())
importance.sort_values(ascending=False).head(25)

## 3. Submission

Retrain with every target week shifted one week later, then score all 1.37M customers.

In [ ]:
preds = pipeline.run_submission(ds, cfg, num_boost_round=model.best_iteration)
sub = submission.build_submission(preds, ds.customers["customer_id"], ds.customer_ids)
sub.to_csv("outputs/submission.csv", index=False)
sub.head()